# UGRP Colab 시뮬레이션
시뮬레이션·렌더링·평가의 기본 실행 노트북입니다. CPU 런타임으로 기본 데모부터 확인합니다.
GPU는 ACT 학습·EGL 렌더링을 선택할 때 사용하며, 물리 계산 전체가 GPU로 옮겨지는 것은 아닙니다.

Chrome **강 / kcm0127@gmail.com** 계정으로 엽니다. Drive 마운트·동기화는 하지 않습니다.
Drive 저장 없이 사용하려면 임시 노트북 https://colab.research.google.com/notebooks/empty.ipynb 에 셀을 복사합니다.
왼쪽 Secrets에 비공개 저장소 읽기 권한이 있는 `UGRP_GITHUB_TOKEN`을 넣고 이 노트북 접근을 허용합니다.
키 값은 셀이나 출력에 입력하지 않습니다. 자동 재접속·런타임 구매는 하지 않습니다.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, uuid
from google.colab import userdata, files
from IPython.display import display, Image

ROOT = Path('/content/ugrp-simulation')
REF = 'codex/colab-simulation'  # 병합 후 main 또는 실험 브랜치로 변경
SIM_ENV = Path('/content/ugrp-sim-env')
PY = str(SIM_ENV / 'bin/python')
BACKEND = 'osmesa'  # CPU 기본값. GPU 런타임에서는 'egl'을 별도 검증 후 사용.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'gh', 'git', 'libosmesa6', 'libegl1', 'libgl1', 'libglfw3', 'ffmpeg', 'fonts-noto-cjk'], check=True)
if ROOT.exists():
    raise RuntimeError('기존 소스/실험을 보존합니다. 새 런타임 또는 다른 ROOT 경로를 사용하세요.')
auth_env = os.environ.copy()
try:
    auth_env['GH_TOKEN'] = userdata.get('UGRP_GITHUB_TOKEN')
    subprocess.run(['gh', 'repo', 'clone', 'kcm0127-dotcom/ugrp', str(ROOT), '--', '--depth', '1', '--branch', REF], env=auth_env, check=True)
finally:
    auth_env.pop('GH_TOKEN', None)
SOURCE_SHA = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('실행 소스 SHA:', SOURCE_SHA)


In [ ]:
# Colab 커널 패키지를 바꾸지 않고 Python 3.12 시뮬레이션 환경을 만듭니다.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--seed', '--python', '3.12', str(SIM_ENV)], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-r', str(ROOT/'requirements-sim.txt'), '-r', str(ROOT/'requirements-test.txt')], check=True)
subprocess.run([PY, '-m', 'pip', 'check'], check=True)
RUN_ENV = os.environ.copy()
RUN_ENV.update(MUJOCO_GL=BACKEND, PYOPENGL_PLATFORM=BACKEND, PYTHONPATH=str(ROOT))

# 모든 작업은 고정한 checkout에서 독립 결과 폴더로 실행합니다.
def run_job(module, args):
    current = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    if current != SOURCE_SHA:
        raise RuntimeError('소스가 바뀌었습니다. 새 실험으로 시작하세요.')
    job = ROOT / 'outputs' / ('colab-' + uuid.uuid4().hex[:12])
    command = [PY, str(ROOT/'scripts/run_colab_simulation.py'), '--output', str(job), '--', PY, '-m', module, *args]
    result = subprocess.run(command, cwd=ROOT, env=RUN_ENV)
    print('종료 코드:', result.returncode, '| 로그:', job/'run.log')
    print('결과 다운로드:', job.with_suffix('.zip'))
    return job, result.returncode


## 1. 무료 기본 물리·카메라 검사
이동 거리 > 0, 12프레임, `ok: true`와 영상을 확인합니다. 자율 운반 성공 검사는 아닙니다.
실행 중 로그는 Colab 파일 패널의 해당 `run.log`에서 확인할 수 있습니다.


In [ ]:
JOB, exit_code = run_job('scripts.sim_quickstart', ['--output', '{output}'])
if exit_code == 0:
    print(json.loads((JOB/'result/summary.json').read_text()))
    display(Image(filename=str(JOB/'result/quickstart.gif')))
else:
    print((JOB/'run.log').read_text()[-6000:])


## 2. 다른 시뮬레이션·평가 실행 — 필요한 작업만
`run_job`에 모듈과 인자를 전달합니다. 출력 인자 위치는 스크립트마다 다릅니다.
예: `JOB, exit_code = run_job('scripts.run_rgb_traffic', ['--scenario', 'crossing', '--out-dir', '{output}'])`.
세 로봇 fixture: `scripts.run_research_dispatch`, `['--output', '{output}', '--variant', 'north_blocked', '--planner', 'fixture', '--motion-smoke']`.

실제 LLM은 **Colab에서 접근 가능한 인증된 모델 endpoint**가 필요합니다. Mac의 `127.0.0.1`은 사용할 수 없습니다.
학습 가중치·원시 데이터가 로컬에만 있으면 파일 업로드 후 경로를 명시해야 합니다.
ACT는 `requirements-reference-act.txt`를 별도 `/content/ugrp-act-env`에 설치하고 `scripts/patch_reference_act.py`를 적용합니다.
시뮬레이션과 ACT의 NumPy 버전이 달라 환경을 합치지 않습니다. 기존 ACT 학습 노트북(PR #84)의 진단·재개 절차를 따릅니다.


## 3. 결과 다운로드 — 실패한 실행도 보존
ZIP에 이 작업의 결과·로그·소스 SHA·패키지·각 파일 해시가 포함됩니다.
Colab VM이 삭제되면 파일도 사라집니다. 실행마다 다운로드하고 로컬에서 해시를 확인하세요.
`files.download` 호출은 로컬 저장 확인을 대신하지 않습니다. ZIP·SHA 파일의 실제 다운로드를 확인합니다.


In [ ]:
# ACT가 필요한 실험에서만 True로 바꾸고 GPU 런타임에서 실행합니다.
INSTALL_ACT = False
if INSTALL_ACT:
    ACT_ENV = Path('/content/ugrp-act-env')
    ACT_PY = str(ACT_ENV/'bin/python')
    subprocess.run(['uv', 'venv', '--seed', '--python', '3.12', str(ACT_ENV)], check=True)
    subprocess.run(['uv', 'pip', 'install', '--python', ACT_PY, '-r', str(ROOT/'requirements-reference-act.txt')], check=True)
    subprocess.run([ACT_PY, '-m', 'pip', 'check'], check=True)
    subprocess.run([ACT_PY, str(ROOT/'scripts/patch_reference_act.py')], cwd=ROOT, check=True)
    subprocess.run([ACT_PY, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())'], check=True)
    # 실제 학습/평가는 해당 실험 프로토콜과 Python 경로 인자를 명시한 별도 작업입니다.


In [ ]:
archive = JOB.with_suffix('.zip')
files.download(str(archive))
files.download(str(archive.with_suffix('.zip.sha256')))


## 중단과 재개
셀 중지 버튼은 소유 세션에 중단 신호를 전달합니다. 아직 실행 중이면 아래 셀로 **그 작업만** 정리합니다.
VM 삭제 전에 완료한 작업별 ZIP을 내려받습니다. 일반 시뮬레이션의 중간 물리 상태 자동 재개는 제공하지 않습니다.
실패한 조건만 새 출력 폴더로 다시 실행하고 전체 조건의 실패·성공을 함께 기록합니다.
학습 checkpoint 재개는 해당 학습기의 지원 범위와 소스·환경·데이터 해시 검증을 따릅니다.


In [ ]:
# 필요할 때만 실행. 다른 작업/런타임은 종료하지 않습니다.
subprocess.run([PY, str(ROOT/'scripts/ugrp_session.py'), 'stop', 'colab-' + JOB.name], cwd=ROOT, check=True)
